# F1 · OLS, Ridge, Lasso, ElasticNet: which one wins, and when

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Compare four linear estimators on four controlled scenarios (number of observations, number of features,
correlation, sparsity) with a repeated simulation, and understand from the numbers, not from the theory, in which scenario
each one wins.

## Dataset

Synthetic, generated by `generate(n, p, rho, k, seed)`: `p` features with correlation `rho^|i−j|`, of which only the first
`k` have a true effect on the target (coefficients between 1 and 3 in absolute value, noise with standard deviation 3).
The four scenarios are in `SCENARIOS`:

| | n | p | correlation | true effects |
|---|---|---|---|---|
| S1 | 200 | 10 | 0 | 10 out of 10 |
| S2 | 50 | 40 | 0 | 5 out of 40 |
| S3 | 100 | 20 | 0.9 | 20 out of 20 |
| S4 | 50 | 40 | 0.7 | 5 out of 40 |

## Educational purpose

Turn "Ridge is fine with multicollinearity, Lasso selects features, ElasticNet is a compromise" from a
memorized sentence into a result you have measured. You will also learn to set up a simulation: data with known truth, many
repetitions, a separate and large enough test set.

## Task

1. Write `evaluate(scenario_name, repetitions=15)`: for every repetition generate `n + 2000` observations with `seed = rep * 10 + 1`, use the first `n` as training and the other 2000 as test, train the four models of `models()` and record a row with `scenario`, `model`, `rmse` (on the test) and `n_nonzero` (number of nonzero coefficients, with `coef_of(model)`). Return a DataFrame.
2. Build `results` by concatenating the four scenarios, and the table `table` with `scenario` on the rows, `model` on the columns and the mean RMSE as value (`pivot_table`).
3. Assign to `winner` the dictionary `{scenario: model with the lowest mean RMSE}`.
4. Build `nonzero`, the table of the mean number of nonzero coefficients (same format as `table`).
5. Assign to `true_false` a dictionary for the three statements:
   - **a**: with n much greater than p and all the effects true (S1) the four methods are practically equivalent;
   - **b**: when p is close to n and few effects are true (S2, S4) OLS is much worse and the Lasso or the ElasticNet win;
   - **c**: Ridge and Lasso set the same number of coefficients to zero.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(n, p, rho, k, seed):
    rng = np.random.default_rng(seed)
    C = rho ** np.abs(np.subtract.outer(np.arange(p), np.arange(p)))       # correlation rho^|i-j|
    X = rng.normal(size=(n, p)) @ np.linalg.cholesky(C).T
    beta = np.zeros(p)
    beta[:k] = rng.choice([-1, 1], k) * rng.uniform(1, 3, k)
    return X, X @ beta + rng.normal(size=n) * 3


SCENARIOS = {"S1": dict(n=200, p=10, rho=0.0, k=10), "S2": dict(n=50, p=40, rho=0.0, k=5),
             "S3": dict(n=100, p=20, rho=0.9, k=20), "S4": dict(n=50, p=40, rho=0.7, k=5)}


def models():
    return {
        "ols": LinearRegression(),
        "ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))),
        "lasso": make_pipeline(StandardScaler(), LassoCV(cv=3, max_iter=5000, random_state=0)),
        "enet": make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.3, 0.7, 0.95], cv=3, max_iter=5000, random_state=0)),
    }


def coef_of(model):
    """Coefficients of the model, even if it sits inside a Pipeline."""
    last = model[-1] if hasattr(model, "steps") else model
    return last.coef_

## Your work

In [ ]:
def evaluate(scenario_name, repetitions=15):
    par = SCENARIOS[scenario_name]
    rows = []
    for rep in range(repetitions):
        ...
    return pd.DataFrame(rows)

In [ ]:
results = pd.concat([evaluate(name) for name in SCENARIOS], ignore_index=True)
table = ...
table.round(2)

In [ ]:
winner = ...
nonzero = ...
nonzero.round(1)

In [ ]:
true_false = {"a": ..., "b": ..., "c": ...}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Inside the loop: `X, y = generate(par["n"] + 2000, par["p"], par["rho"], par["k"], seed=rep * 10 + 1)`, then `X_tr, y_tr, X_te, y_te = X[:par["n"]], y[:par["n"]], X[par["n"]:], y[par["n"]:]`. Then `for name, m in models().items(): m.fit(X_tr, y_tr)` and `mean_squared_error(y_te, m.predict(X_te), squared=False)`.'):
    check(len(results) == 4 * 15 * 4, "results has one row per scenario, repetition and model",
          "you need 4 scenarios x 15 repetitions x 4 models = 240 rows")
    check(list(table.index) == ["S1", "S2", "S3", "S4"] and set(table.columns) == {"ols", "ridge", "lasso", "enet"},
          "the table has scenarios on the rows and models on the columns", "use pivot_table(index='scenario', columns='model', values='rmse')")
    check(table.loc["S1"].max() / table.loc["S1"].min() < 1.03,
          "in S1 the four methods are equivalent (difference < 3%)", "in S1 (large n, small p) the methods should give the same RMSE")
    check(all(table.loc[s, "ols"] > 1.4 * table.loc[s].min() for s in ["S2", "S4"]),
          "in S2 and S4 OLS is much worse than the best (at least 1.4 times)",
          "with p close to n OLS should have a much higher RMSE than the best")
    check(winner == table.idxmin(axis=1).to_dict(), "winner consistent with the table", "winner = table.idxmin(axis=1).to_dict()")
    check(winner["S2"] in ("lasso", "enet") and winner["S4"] in ("lasso", "enet"),
          "in S2 and S4 the methods that zero the coefficients win", "in S2 and S4 the winner should be lasso or enet")
    check(nonzero.loc["S2", "ols"] == 40 and nonzero.loc["S2", "ridge"] == 40,
          "OLS and Ridge never zero anything", "OLS and Ridge must have all 40 coefficients nonzero")
    check(nonzero.loc["S2", "lasso"] < 30, "the Lasso in S2 keeps many fewer than 40 features",
          "the Lasso should zero a good part of the 40 coefficients in S2")
    check(true_false == {"a": True, "b": True, "c": False}, "statements judged correctly",
          "look at table and nonzero: Ridge zeroes nothing")
done()

---
The commented solution is in `solutions/F_compare/F1_ols_ridge_lasso_elasticnet_simulation.ipynb`. Open it after you pass the check, or after you have really tried.